# PR-CLIP v4 — Reliability-Aware Bidirectional Reconstruction

This notebook runs the supplied PR-CLIP pipeline end-to-end for **one epoch**, **batch size 16**, with pretrained **GeoRSCLIP RET-2** weights. It trains on the bundled RET-2 captions, evaluates RSICD retrieval, and saves the run/checkpoint to Google Drive.

Before running: put `PR_CLIP_v4_Source.zip` in the root of Google Drive, or upload it when prompted. The notebook downloads the image archive and RET-2 checkpoint. The architecture is an experimental proposal; the notebook does not promise a Mean Recall gain.


In [ ]:
from google.colab import drive
from pathlib import Path
import shutil, zipfile, os

drive.mount('/content/drive')
source_zip = Path('/content/drive/MyDrive/PR_CLIP_v4_Source.zip')
if not source_zip.exists():
    from google.colab import files
    uploaded = files.upload()
    candidates = [Path('/content') / name for name in uploaded if name.lower().endswith('.zip')]
    if not candidates:
        raise FileNotFoundError('Upload PR_CLIP_v4_Source.zip to continue.')
    source_zip = candidates[0]

repo_dir = Path('/content/PR-CLIP')
if repo_dir.exists():
    shutil.rmtree(repo_dir)
with zipfile.ZipFile(source_zip) as zf:
    zf.extractall('/content')
if not (repo_dir / 'training/main.py').exists():
    raise FileNotFoundError('The source ZIP must contain PR-CLIP/training/main.py')
os.chdir(repo_dir)
print('Source ready:', repo_dir)


In [ ]:
%cd /content/PR-CLIP
!pip -q install open_clip_torch==2.20.0 clip-benchmark==1.6.1 transformers==4.45.2 adapters timm sentence-transformers wandb pyyaml gdown
!pip -q install 'protobuf>=6.31.1,<7'
!pip -q install cn_clip --no-deps
!pip -q install faiss-cpu click


In [ ]:
%cd /content/PR-CLIP
!gdown --fuzzy 'https://drive.google.com/file/d/15htMe2iE4kVujjWURxfCxZXk8vAAetoT/view' -O /content/rsicd_images.tar.gz
!mkdir -p images
!tar -xzf /content/rsicd_images.tar.gz -C images

# Flatten the archive because the PR-CLIP CSVs store image basenames.
from pathlib import Path
import shutil
image_root = Path('images')
for p in list(image_root.rglob('*')):
    if p.is_file() and p.parent != image_root:
        target = image_root / p.name
        if target.exists():
            raise RuntimeError(f'Duplicate image basename in archive: {p.name}')
        shutil.move(str(p), str(target))
for p in sorted(image_root.rglob('*'), reverse=True):
    if p.is_dir() and not any(p.iterdir()):
        p.rmdir()
Path('cache/weights').mkdir(parents=True, exist_ok=True)


In [ ]:
%cd /content/PR-CLIP
!wget -O cache/weights/RET-2.pt 'https://huggingface.co/Zilun/GeoRSCLIP/resolve/main/ckpt/RS5M_ViT-B-32_RET-2.pt'

from pathlib import Path
weight_path = Path('cache/weights/RET-2.pt')
if not weight_path.exists() or weight_path.stat().st_size < 100_000_000:
    raise RuntimeError('RET-2 checkpoint download is missing or incomplete.')
print(f'RET-2 checkpoint: {weight_path.stat().st_size / 1e6:.1f} MB')


In [ ]:
%cd /content/PR-CLIP
import os
import pandas as pd
for csv_path in ['dataset/train/ret2.csv', 'dataset/test/rsicd_test.csv']:
    if not Path(csv_path).is_file():
        raise FileNotFoundError(csv_path)
    names = pd.read_csv(csv_path)['filename'].unique()
    missing = [name for name in names if not (Path('images') / name).exists()]
    print(f'{csv_path}: {len(names)} unique images; missing={len(missing)}')
    if missing:
        raise FileNotFoundError(f'{len(missing)} images missing; first missing: {missing[:5]}')


## Train and evaluate

This launches the original training entry point with the notebook's v3 settings: batch size 16, one epoch, AMP, RET-2 initialization, and RSICD retrieval evaluation every epoch. The reconstruction branch uses a 100× learning-rate scale while the backbone keeps the configured `2e-7` base LR.


In [ ]:
%cd /content/PR-CLIP
import subprocess, sys

RUN_NAME = 'rsicd_v4_reliability_bidir_ep1_bs16_lr2e7'
LOG_DIR = '/content/drive/MyDrive/prclip_v4_logs'
cmd = [sys.executable, '-m', 'training.main',
 '--train-data', './dataset/train/ret2.csv', '--images-dir', './images',
 '--retrieval-data', './dataset/test/rsicd_test.csv', '--retrieval-images-dir', './images',
 '--csv-separator', ',', '--csv-img-key', 'filename',
 '--retrieval-csv-separator', ',', '--retrieval-csv-img-key', 'filename',
 '--retrieval-frequency', '1', '--save-frequency', '0', '--save-most-recent',
 '--epochs', '1', '--batch-size', '16', '--workers', '2', '--precision', 'amp',
 '--lr', '2e-7', '--warmup', '100', '--weight_decay', '0.5', '--max-grad-norm', '50',
 '--image-model', 'ViT-B-32', '--image-model-builder', 'openclip',
 '--text-model', 'ViT-B-32', '--text-model-builder', 'openclip',
 '--pretrained-image-model', '--pretrained-text-model', '--load-clip-weights',
 '--load-clip-weights-from', './cache/weights/RET-2.pt',
 '--loss', 'InfoNCE', '--seed', '1', '--name', RUN_NAME, '--logs', LOG_DIR]
print('Launching one-epoch PR-CLIP v4 run:')
print(' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
from pathlib import Path
run_dir = Path(LOG_DIR) / RUN_NAME
if not run_dir.exists():
    raise FileNotFoundError(f'Run output not found: {run_dir}')
checkpoints = list(run_dir.rglob('*.pt'))
print('Checkpoint files:', [(p.name, round(p.stat().st_size / 1e6, 1)) for p in checkpoints])
best = [p for p in checkpoints if p.name == 'best.pt']
if not best:
    raise FileNotFoundError('Training/evaluation finished without best.pt; inspect the log above.')
print('Best checkpoint:', best[0])
print('Run artifacts and retrieval logs:', run_dir)
